# Build a FAISS Vector DB from `RAG_Data`

Pipeline: **load metadata + scraped text → join on `Document_ID` → chunk → embed → `vector_df` → FAISS index → save → query**

Metadata schema (one JSON per item):
`Document_ID`, `Source_Type`, `Source_Name`, `URL`, `Title`, `Published_Date`, `Location`

Directory layout:
```
RAG_Data/
├── metadata/{blogs_forums, youtube_podcast}/*.json
└── scraped_text/{blogs_forums, youtube_podcast}/   # body, joined by Document_ID
```

## 0. Install (run once)

In [ ]:
# !pip install -q faiss-cpu sentence-transformers pandas numpy pyarrow

## 1. Imports & configuration
Edit `DATA_ROOT` to point at your `RAG_Data` folder.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd

# >>> EDIT THIS PATH <<<
DATA_ROOT     = Path("/Users/mahiwashim/Documents/Project/Gulf FEI/Data/RAG_Data")

METADATA_DIR  = DATA_ROOT / "metadata"
TEXT_DIR      = DATA_ROOT / "scraped_text"
SOURCES       = ["blogs_forums", "youtube_podcast"]   # sub-folders under each tree

EMBED_MODEL   = "BAAI/bge-small-en-v1.5"   # 384-dim, fast, strong baseline
CHUNK_SIZE    = 800                         # characters per chunk
CHUNK_OVERLAP = 120                         # sliding overlap between chunks
OUT_DIR       = Path("rag_artifacts")       # where the FAISS DB is written
OUT_DIR.mkdir(parents=True, exist_ok=True)

META_FIELDS = ["Document_ID","Source_Type","Source_Name","URL","Title","Published_Date","Location"]
TEXT_KEYS   = ["text","content","body","transcript","clean_text","raw_text"]  # for JSON-bodied text
assert DATA_ROOT.is_dir(), f"Not found: {DATA_ROOT}"
print("Data root OK:", DATA_ROOT)

## 2. Loaders

`load_metadata` reads each JSON into a record keyed by `Document_ID`.
`load_texts` reads the scraped body (`.txt` / `.md`, or a JSON with a text field), keyed by file stem (= `Document_ID`).

In [ ]:
def load_metadata(meta_dir: Path) -> dict:
    """{Document_ID: {field: value, ...}} for one source folder."""
    recs = {}
    meta_dir = Path(meta_dir)
    if not meta_dir.is_dir():
        print("  ! missing:", meta_dir); return recs
    for fp in sorted(meta_dir.glob("*.json")):
        try:
            data = json.loads(fp.read_text(encoding="utf-8", errors="ignore"))
        except Exception as e:
            print("  ! bad json", fp.name, e); continue
        for d in (data if isinstance(data, list) else [data]):
            did = str(d.get("Document_ID") or fp.stem)
            rec = {k: d.get(k, "") for k in META_FIELDS}
            rec["Document_ID"] = did
            recs[did] = rec
    return recs

def load_texts(text_dir: Path) -> dict:
    """{Document_ID: body_text} for one source folder."""
    texts = {}
    text_dir = Path(text_dir)
    if not text_dir.is_dir():
        print("  ! missing:", text_dir); return texts
    for fp in sorted(text_dir.iterdir()):
        if fp.name.startswith(".") or not fp.is_file():
            continue
        ext = fp.suffix.lower()
        if ext in {".txt", ".md", ".markdown"}:
            texts[fp.stem] = fp.read_text(encoding="utf-8", errors="ignore").strip()
        elif ext in {".json", ".jsonl"}:
            raw = fp.read_text(encoding="utf-8", errors="ignore")
            data = []
            try:
                data = json.loads(raw)
                data = data if isinstance(data, list) else [data]
            except json.JSONDecodeError:
                data = [json.loads(l) for l in raw.splitlines() if l.strip()]
            for d in data:
                key = str(d.get("Document_ID") or fp.stem)
                body = next((d[k] for k in TEXT_KEYS if d.get(k)), "")
                texts[key] = str(body).strip()
        else:  # unknown extension -> treat as raw text
            texts[fp.stem] = fp.read_text(encoding="utf-8", errors="ignore").strip()
    return texts

## 3. Load & join all sources on `Document_ID`

In [ ]:
documents = []
for src in SOURCES:
    meta  = load_metadata(METADATA_DIR / src)
    texts = load_texts(TEXT_DIR / src)
    matched = 0
    for did, rec in meta.items():
        body = texts.get(did, "")
        if not body:
            continue                      # skip items with no scraped text
        matched += 1
        documents.append({**rec, "Source_Folder": src, "text": body})
    only_text = set(texts) - set(meta)
    print(f"[{src:16s}] meta={len(meta):4d}  text={len(texts):4d}  joined={matched:4d}"
          f"  (text w/o metadata: {len(only_text)})")

print(f"\nTotal joined documents: {len(documents)}")
assert documents, "No documents joined — check that text filenames match Document_ID."
docs_df = pd.DataFrame(documents)
docs_df[["Document_ID","Source_Type","Source_Name","Title","Published_Date","Location"]].head()

## 4. Chunking
Recursive character splitter (paragraph → line → sentence → word) with overlap, so chunks stay clean and context survives the seams.

In [ ]:
def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r"[ \t]+", " ", text).strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]
    separators = ["\n\n", "\n", ". ", " "]
    def _split(s, seps):
        if len(s) <= chunk_size or not seps:
            return [s]
        sep = seps[0]; parts = s.split(sep); out=[]; cur=""
        for p in parts:
            piece = p + sep
            if len(cur)+len(piece) <= chunk_size:
                cur += piece
            else:
                if cur: out.append(cur.strip())
                if len(piece) > chunk_size:
                    out.extend(_split(p, seps[1:])); cur=""
                else:
                    cur = piece
        if cur.strip(): out.append(cur.strip())
        return [c for c in out if c]
    raw = _split(text, separators)
    if overlap <= 0:
        return raw
    stitched=[]
    for i,c in enumerate(raw):
        stitched.append(c if i==0 else (raw[i-1][-overlap:] + " " + c).strip())
    return stitched

# smoke test
print(len(chunk_text(documents[0]["text"])), "chunks from doc 0")

## 5. Build the `vector_df` (one row per chunk)
Metadata is carried onto every chunk so retrieval results are fully attributable.

In [ ]:
rows = []
for doc in documents:
    for i, ch in enumerate(chunk_text(doc["text"])):
        rows.append({
            "chunk_id":       f"{doc['Document_ID']}#{i}",
            "Document_ID":    doc["Document_ID"],
            "chunk_index":    i,
            "Source_Type":    doc.get("Source_Type",""),
            "Source_Name":    doc.get("Source_Name",""),
            "Source_Folder":  doc.get("Source_Folder",""),
            "Title":          doc.get("Title",""),
            "URL":            doc.get("URL",""),
            "Published_Date": doc.get("Published_Date",""),
            "Location":       doc.get("Location",""),
            "text":           ch,
        })

vector_df = pd.DataFrame(rows).reset_index(drop=True)
print(f"vector_df: {len(vector_df)} chunks from {vector_df['Document_ID'].nunique()} documents")
vector_df.head()

## 6. Embed the chunks
First run downloads the model (~120 MB). Vectors are L2-normalized so inner product == cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = model.get_sentence_embedding_dimension()

embeddings = model.encode(
    vector_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print("embeddings:", embeddings.shape, "| dim:", EMBED_DIM)
assert embeddings.shape[0] == len(vector_df)

## 7. Build the FAISS index
`IndexFlatIP` over normalized vectors = exact cosine search. `IndexIDMap2` keeps each vector's row id aligned to `vector_df`.

In [ ]:
import faiss

base  = faiss.IndexFlatIP(EMBED_DIM)
index = faiss.IndexIDMap2(base)
ids   = np.arange(len(vector_df)).astype("int64")
index.add_with_ids(embeddings, ids)
print("FAISS index built:", index.ntotal, "vectors, dim", EMBED_DIM)

## 8. Save the vector DB

In [ ]:
faiss.write_index(index, str(OUT_DIR / "index.faiss"))
np.save(OUT_DIR / "vectors.npy", embeddings)
vector_df.to_parquet(OUT_DIR / "vector_df.parquet", index=False)

config = {
    "model_name": EMBED_MODEL,
    "dim": int(EMBED_DIM),
    "metric": "cosine (inner product on L2-normalized vectors)",
    "num_documents": int(vector_df["Document_ID"].nunique()),
    "num_chunks": int(len(vector_df)),
    "chunk_size": CHUNK_SIZE,
    "overlap": CHUNK_OVERLAP,
    "sources": SOURCES,
    "files": {"index": "index.faiss", "vectors": "vectors.npy", "metadata": "vector_df.parquet"},
}
(OUT_DIR / "build_config.json").write_text(json.dumps(config, indent=2))
print("Saved to", OUT_DIR.resolve())
for f in ["index.faiss","vector_df.parquet","vectors.npy","build_config.json"]:
    print("  -", f)

## 9. Query / retrieval (this is what you deploy)

In [ ]:
def search(query: str, k: int = 5, source_folder: str | None = None, oversample: int = 6):
    n = min(k*oversample if source_folder else k, index.ntotal)
    qv = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idx = index.search(qv, n)
    hits = []
    for s, row_id in zip(scores[0], idx[0]):
        if row_id == -1:
            continue
        r = vector_df.iloc[int(row_id)]
        if source_folder and r["Source_Folder"] != source_folder:
            continue
        hits.append({
            "score": round(float(s), 4),
            "Document_ID": r["Document_ID"],
            "Source_Type": r["Source_Type"],
            "Title": r["Title"],
            "URL": r["URL"],
            "Location": r["Location"],
            "text": r["text"],
        })
        if len(hits) >= k:
            break
    return hits

for h in search("your test question here", k=5):
    print(f"[{h['score']}] {h['Source_Type']:8s} | {h['Title']}")
    print("   ", h["text"][:160], "...\n")

## 10. (Optional) Reload later without rebuilding

```python
import faiss, json, pandas as pd
from sentence_transformers import SentenceTransformer
from pathlib import Path

OUT_DIR = Path("rag_artifacts")
cfg = json.loads((OUT_DIR/"build_config.json").read_text())
index = faiss.read_index(str(OUT_DIR/"index.faiss"))
vector_df = pd.read_parquet(OUT_DIR/"vector_df.parquet")
model = SentenceTransformer(cfg["model_name"])
# then reuse the search() function above
```

**Scaling note:** `IndexFlatIP` is exact and fine to ~1M chunks. Beyond that, swap in `IndexHNSWFlat` or a trained `IndexIVFFlat` in step 7.